# Power BI 重新設計規格書 — 04_Inventory_Performance_Analysis

> **專案背景**：資料來源為 PwC × Kaggle Inventory Analysis，透過 PostgreSQL ELT 建立 Raw → Staging → Marts 三層架構，最終產出 Star Schema（`dim_product`、`dim_store`、`dim_vendor`、`dim_date`、`fact_sales`、`fact_inventory_snapshot`），再以 Power BI Import Mode 消費。目標 KPI 為 Inventory Turnover、DSI、Stockout Rate、Dead Stock、ABC Classification 與 Reorder Point。

***

## 一、資料約束與設計前提

在動手做任何 Visual 之前，先釐清資料的真實限制，避免設計出無法支撐的 KPI：

| 資料表 | 時間點/範圍 | 說明 |
|---|---|---|
| `raw_beg_inventory` | 期初一個快照（2016-01-01）| 約 20 萬筆 |
| `raw_end_inventory` | 期末一個快照（2016-12-31）| 約 20 萬筆 |
| `raw_sales` | 2016-01-01 ~ 2016-12-31，完整 366 天 | 1,200 萬+ 筆 |
| `fact_inventory_snapshot` | BEGINNING + ENDING 兩列 per SKU/Store | 不能計算中間月份庫存趨勢 |

**關鍵限制**：`fact_inventory_snapshot` 只有期初與期末兩個時間點，因此「月度庫存趨勢線」在本資料集中**無法精準計算**，設計時必須誠實面對這個限制。但 `fact_sales` 有完整 366 天，可計算月度銷售趨勢、月均銷售量、Reorder Point 等。

***

## 二、資料模型（Power BI Relationship 設定）

Power BI Import Mode 下，嚴格遵守 Star Schema 單向篩選，避免雙向 Cross-filter 造成 Ambiguity 問題：

| From Table (Many) | From Column | To Table (One) | To Column | Cardinality | Direction |
|---|---|---|---|---|---|
| `fact_sales` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single → |
| `fact_sales` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single → |
| `fact_sales` | `vendor_sk` | `dim_vendor` | `vendor_sk` | Many-to-One | Single → |
| `fact_sales` | `sales_date` | `dim_date` | `date_key` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single → |
| `fact_inventory_snapshot` | `snapshot_date` | `dim_date` | `date_key` | Many-to-One | Single → |

**重要設定**：`dim_date` 建立後，在 Power BI Model View 右鍵 → **Mark as Date Table**（Date column = `date_key`），否則 `DATESYTD`、`DATEADD` 等 Time Intelligence 函數不會生效。

***

## 三、完整 DAX Measure 規格

所有 Measures 建議集中在一張獨立 `_Measures` 表（新增空白表，不 import 任何資料），方便管理與 GitHub 文件化。

### Group 01：基礎數值

```dax
-- 總銷售額
Total Revenue =
SUMX('fact_sales', 'fact_sales'[sales_dollars])

-- 總銷售量
Total Sales Qty =
SUM('fact_sales'[sales_quantity])

-- 估算 COGS（銷售量 × 成本價）
Total COGS =
SUMX('fact_sales', 'fact_sales'[estimated_cogs])

-- 期初庫存總值
Beginning Inventory Value =
CALCULATE(
    SUM('fact_inventory_snapshot'[total_inventory_value]),
    'fact_inventory_snapshot'[snapshot_type] = "BEGINNING"
)

-- 期末庫存總值
Ending Inventory Value =
CALCULATE(
    SUM('fact_inventory_snapshot'[total_inventory_value]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)

-- 期末庫存量（在手）
Current Stock Qty =
CALCULATE(
    SUM('fact_inventory_snapshot'[quantity_on_hand]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)

-- 平均庫存值（期初+期末 / 2）
Average Inventory Value =
DIVIDE(
    [Beginning Inventory Value] + [Ending Inventory Value],
    2
)
```

### Group 02：庫存周轉 KPI

```dax
-- 庫存周轉率（年度）
Inventory Turnover =
DIVIDE(
    [Total COGS],
    [Average Inventory Value],
    BLANK()
)

-- DSI（庫存銷售天數）
Days Sales of Inventory (DSI) =
VAR _turnover = [Inventory Turnover]
RETURN
    IF(
        _turnover = 0 || ISBLANK(_turnover),
        BLANK(),
        DIVIDE(365, _turnover)
    )

-- YTD 版本（搭配 dim_date Time Intelligence）
Inventory Turnover YTD =
CALCULATE(
    [Inventory Turnover],
    DATESYTD('dim_date'[date_key])
)
```

> ⚠️ **資料限制提示**：由於庫存快照只有期初/期末兩個時間點，`Inventory Turnover` 與 `DSI` 在月度視圖下會顯示相同值（分母 Average Inventory 不隨月份變動）。建議在 Page 1 的圖表 subtitle 加入說明，或限制此 KPI 只在年度層級顯示。

### Group 03：缺貨率

```dax
-- 缺貨快照記錄數
Out of Stock Records =
CALCULATE(
    COUNTROWS('fact_inventory_snapshot'),
    'fact_inventory_snapshot'[quantity_on_hand] <= 0
)

-- 總快照記錄數
Total Snapshot Records =
COUNTROWS('fact_inventory_snapshot')

-- 缺貨率（以快照比例定義）
Stockout Rate =
DIVIDE([Out of Stock Records], [Total Snapshot Records], BLANK())
```

### Group 04：呆滯庫存

```dax
-- 過去 90 天銷售量（以 dim_date 為基準）
Sales Qty Last 90 Days =
CALCULATE(
    SUM('fact_sales'[sales_quantity]),
    DATESINPERIOD(
        'dim_date'[date_key],
        MAX('dim_date'[date_key]),
        -90,
        DAY
    )
)

-- 呆滯庫存金額（90 天無銷售 + 期末有庫存）
Dead Stock Value (90 Days) =
CALCULATE(
    [Ending Inventory Value],
    FILTER(
        'dim_product',
        [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days])
    ),
    'fact_inventory_snapshot'[quantity_on_hand] > 0
)

-- 呆滯庫存 SKU 數
Dead Stock SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        ( [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days]) )
        &&
        CALCULATE(
            SUM('fact_inventory_snapshot'[quantity_on_hand]),
            'fact_inventory_snapshot'[snapshot_type] = "ENDING"
        ) > 0
    )
)

-- 呆滯庫存占比（占期末總庫存金額）
Dead Stock % of Total Inventory =
DIVIDE([Dead Stock Value (90 Days)], [Ending Inventory Value], BLANK())

-- 呆滯旗標（表格用）
Dead Stock Flag =
IF([Dead Stock Value (90 Days)] > 0, "Dead Stock", "Active")
```

> ⚠️ **資料限制說明**：本資料集期末快照日期為 2016-12-31，因此「過去 90 天」的計算實際上是 2016-10-03 ~ 2016-12-31。任何篩選上下文改變 `MAX('dim_date'[date_key])` 時，90 天視窗會跟著移動，邏輯正確。

### Group 05：再訂購點

```dax
-- 日均銷售量
Avg Daily Sales Qty =
VAR _totalQty   = SUM('fact_sales'[sales_quantity])
VAR _activeDays =
    CALCULATE(
        DISTINCTCOUNT('dim_date'[date_key]),
        CROSSFILTER('fact_sales'[sales_date], 'dim_date'[date_key], BOTH)
    )
RETURN
    DIVIDE(_totalQty, _activeDays, BLANK())

-- 日銷售量標準差（統計安全庫存用）
Stddev Daily Sales Qty =
VAR _salesByDay =
    ADDCOLUMNS(
        VALUES('dim_date'[date_key]),
        "@DailySales",
        CALCULATE(SUM('fact_sales'[sales_quantity]))
    )
VAR _avgDaily = AVERAGEX(_salesByDay, [@DailySales])
VAR _variance =
    AVERAGEX(
        _salesByDay,
        ([@DailySales] - _avgDaily) ^ 2
    )
RETURN SQRT(_variance)

-- 安全庫存量（Z=1.65, 95% 服務水準, Lead Time=7 天）
Safety Stock Qty =
VAR _leadTimeDays = 7
VAR _zScore       = 1.65
RETURN
    _zScore * [Stddev Daily Sales Qty] * SQRT(_leadTimeDays)

-- 再訂購點數量
Reorder Point Qty =
VAR _leadTimeDays       = 7
VAR _demandDuringLead   = [Avg Daily Sales Qty] * _leadTimeDays
RETURN
    ROUND(_demandDuringLead + [Safety Stock Qty], 0)

-- 補貨警示（四色旗標）
Reorder Alert =
VAR _currentStock = [Current Stock Qty]
VAR _rop          = [Reorder Point Qty]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_rop) || ISBLANK(_currentStock), BLANK(),
        _currentStock <= 0,          "🔴 Out of Stock",
        _currentStock <= _rop,       "🟡 Reorder Now",
        _currentStock <= _rop * 1.2, "🟠 Low Stock",
        "🟢 OK"
    )

-- 需補貨 SKU 總數（Page 3 Card 用）
Reorder SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        [Reorder Alert] = "🟡 Reorder Now"
    )
)
```

### Group 06：ABC 輔助欄位（Calculated Columns on `dim_product`）

以下為 Calculated Column（不是 Measure），直接加在 `dim_product` 表：

```dax
-- 顯示用欄位（消除 Blank 軸標籤）
ABC Class Display =
COALESCE('dim_product'[abc_class], "Unclassified")

-- 排序欄位（設定 "Sort by column" = 此欄）
ABC Class Sort =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", 1,
    'dim_product'[abc_class] = "B", 2,
    'dim_product'[abc_class] = "C", 3,
    4  -- Unclassified 排最後
)

-- 業務語言版本（Page 3 矩陣用）
ABC Priority Group =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", "A - Critical",
    'dim_product'[abc_class] = "B", "B - Important",
    'dim_product'[abc_class] = "C", "C - Long Tail",
    "Review - Unclassified"
)
```

### Group 07：未分類庫存監控

```dax
Unclassified SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)

Unclassified Inventory Value =
CALCULATE(
    [Ending Inventory Value],
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)

Unclassified Dead Stock Value =
CALCULATE(
    [Dead Stock Value (90 Days)],
    FILTER('dim_product', ISBLANK('dim_product'[abc_class]))
)
```

### Group 08：動態圖表標題

```dax
Title - Dead Stock by ABC =
VAR _store = SELECTEDVALUE('dim_store'[store_number], "All Stores")
VAR _year  = SELECTEDVALUE('dim_date'[year], "All Years")
RETURN
    "Dead Stock Value (90 Days) by ABC Class | Store: " & _store & " | Year: " & _year

Title - Reorder Matrix =
VAR _abc   = SELECTEDVALUE('dim_product'[abc_class], "All Classes")
VAR _store = SELECTEDVALUE('dim_store'[store_number], "All Stores")
RETURN
    "Replenishment Status | ABC: " & _abc & " | Store: " & _store
```

***

## 四、三頁報表設計規格

### 全域設計原則

- **Slicer 組合（所有頁面一致）**：`Year`、`Month`、`Store`、`Vendor`、`ABC Class`；Page 2 & 3 額外加 `Product Search`（Search Slicer 類型）
- **色彩規則**：正常用中性灰藍，警示僅用少量強調色：🔴 Out of Stock / 🟡 Reorder Now / 🟠 Low Stock / 🟢 OK
- **導航按鈕**：每頁右上角固定放 Page 1 / Page 2 / Page 3 按鈕 + `Last Refresh Date` Card
- **Drill-through 設定**：Page 1 點擊 Store → Drill-through 到 Page 2；點擊 Product → Drill-through 到 Page 3

***

### Page 1 — Executive Overview

**核心問題**：*整體庫存績效健不健康？哪個方向最值得優先追蹤？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav: 1 Overview | 2 Inventory Risk | 3 Replenishment]   [Refresh]  │
│ Title: Inventory Performance Overview · FY 2016                      │
│ Slicers: Year | Month | Store | Vendor | ABC Class                   │
├──────────────────────────────────────────────────────────────────────┤
│  KPI Card 1       │ KPI Card 2       │ KPI Card 3    │ KPI Card 4   │
│  Inventory        │ DSI              │ Stockout Rate │ Dead Stock % │
│  Turnover         │ (Days)           │ (%)           │ of Inventory │
├──────────────────────────────────────────────────────────────────────┤
│  Line: Monthly Total Revenue & COGS Trend  (fact_sales 有完整366天) │
├────────────────────────────────────┬─────────────────────────────────┤
│  Column: Stockout Rate by Store    │ Bar: Dead Stock Value by ABC    │
│  (排序高→低, 抓風險門店)            │ (突出 C 類積壓)                 │
├────────────────────────────────────┴─────────────────────────────────┤
│  Matrix: Store × ABC Class / Revenue / End. Inv / Turnover / DSI    │
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | Axis / Legend | Values | 備註 |
|---|---|---|---|---|
| KPI Card 1 | Card (New) | — | `Inventory Turnover` | Subtitle: "FY 2016 Estimate" |
| KPI Card 2 | Card | — | `Days Sales of Inventory (DSI)` | Format: 0 days |
| KPI Card 3 | Card | — | `Stockout Rate` | Format: % |
| KPI Card 4 | Card | — | `Dead Stock % of Total Inventory` | 目標線 5%，紅色警示 |
| Monthly Trend | Line | `dim_date[month_name_short]` | `Total Revenue`, `Total COGS` | 雙線，Revenue 藍 / COGS 橙 |
| Stockout by Store | Clustered Column | `dim_store[store_number]` | `Stockout Rate` | 依高→低排序 |
| Dead Stock by ABC | Bar | `dim_product[ABC Class Display]` | `Dead Stock Value (90 Days)` | 使用動態 Title Measure |
| Summary Matrix | Matrix | Rows: `Store`, `ABC Class Display` | Revenue / End. Inv Value / Turnover / DSI / Stockout Rate | Conditional Format: Turnover 低→紅 |

> **設計決策說明**：月度趨勢選擇 Revenue + COGS（來自 `fact_sales`），而非 Inventory Turnover，因為後者的分母庫存值只有年初/年末兩點，月度顯示無意義。誠實呈現資料限制是作品集的加分項。

***

### Page 2 — Inventory Risk Analysis

**核心問題**：*哪些商品或門店同時面臨缺貨與呆滯的雙重風險？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav] [Refresh]                                                      │
│ Title: Inventory Risk Analysis                                       │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product Search │
├──────────────────────────────────────────────────────────────────────┤
│  Card: Dead Stock Value  │ Card: Dead Stock SKU # │ Card: Out of Stock Records    │
├───────────────────────────────────────┬──────────────────────────────┤
│  Scatter: Stockout Rate vs            │ Treemap/Bar:                 │
│  Dead Stock % (by Store or Category) │ Dead Stock by Vendor         │
├───────────────────────────────────────┴──────────────────────────────┤
│  Detail Table: Product / Store / ABC / QOH / Sales90D / Value / Flag│
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | 設定 |
|---|---|---|
| Dead Stock Value Card | Card | `Dead Stock Value (90 Days)` |
| Dead Stock SKU Count Card | Card | `Dead Stock SKU Count` |
| OOS Records Card | Card | `Out of Stock Records` |
| Risk Scatter | Scatter | X: `Stockout Rate` / Y: `Dead Stock % of Total Inventory` / Size: `Ending Inventory Value` / Details: `dim_store[store_number]` |
| Dead Stock by Vendor | Bar | Axis: `dim_vendor[vendor_name]` / Value: `Dead Stock Value (90 Days)` / Top N: 15 |
| Risk Detail Table | Table | `dim_product[brand]` / `dim_store[store_number]` / `ABC Class Display` / `Current Stock Qty` / `Sales Qty Last 90 Days` / `Dead Stock Value (90 Days)` / `Dead Stock Flag` |

**Conditional Formatting for Detail Table**：
- `Dead Stock Flag` = "Dead Stock" → 背景色紅（#FFE0E0）
- `Current Stock Qty` = 0 → 字體色紅

***

### Page 3 — Replenishment & ABC Prioritization

**核心問題**：*哪些 SKU 現在該補貨，按 ABC 優先順序如何排？*

#### Wireframe

```
┌──────────────────────────────────────────────────────────────────────┐
│ [Nav] [Refresh]                                                      │
│ Title: [Title - Reorder Matrix 動態標題]                             │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product Search │
├──────────────────────────────────────────────────────────────────────┤
│  Card: Reorder SKU Count │ Card: Avg Daily Sales │ Card: End. Inv $ │
├───────────────────────────────────────┬──────────────────────────────┤
│  Bar: Reorder Alert Count by ABC      │ Bar: Top 15 Reorder Products │
│  (🟡+🔴 的 SKU 數，按 ABC 分組)      │ (Filter: Reorder Now only)   │
├───────────────────────────────────────┴──────────────────────────────┤
│  Matrix: Product / Store / ABC / QOH / AvgDaily / Safety / ROP /    │
│          Reorder Alert / Vendor                                       │
└──────────────────────────────────────────────────────────────────────┘
```

#### Visual 規格

| Visual | Type | 設定 |
|---|---|---|
| Reorder SKU Count | Card | `Reorder SKU Count` |
| Avg Daily Sales | Card | `Avg Daily Sales Qty` / Format: 0.0 |
| Ending Inventory Value | Card | `Ending Inventory Value` |
| Alert by ABC | Clustered Column | Axis: `ABC Class Display` / Value: `Reorder SKU Count` / Legend: `Reorder Alert` / Sort: `ABC Class Sort` |
| Top Reorder Products | Bar | Axis: `dim_product[brand]` / Value: `Reorder Point Qty` / Visual Level Filter: `Reorder Alert = "🟡 Reorder Now"` / Top N: 15 |
| Main Replenishment Matrix | Matrix | Rows: `ABC Priority Group`, `brand`, `store_number` / Values: `Current Stock Qty`, `Avg Daily Sales Qty`, `Safety Stock Qty`, `Reorder Point Qty`, `Reorder Alert` / Conditional Format: Reorder Alert 依值設色 |

**矩陣 Conditional Formatting 規則**（`Reorder Alert` 欄）：
```
🔴 Out of Stock  → 背景 #FF0000, 字 白色
🟡 Reorder Now   → 背景 #FFD700, 字 黑色
🟠 Low Stock     → 背景 #FFA500, 字 黑色
🟢 OK            → 背景 #E8F5E9, 字 黑色
```

***

## 五、Tooltip 設計

高價值 Visual 建議加上 Report Page Tooltip，讓使用者 hover 時看到 SKU 層級細節：

### Tooltip Page — SKU Detail（5×3 cm 小頁面）

Visuals：
- `ABC Priority Group`（Text）
- `Avg Daily Sales Qty`（Card）
- `Safety Stock Qty`（Card）
- `Reorder Point Qty`（Card）
- `Current Stock Qty`（Card）
- `Reorder Alert`（Card）

在 Page 3 矩陣的 `brand` 欄位設定此 Tooltip Page。

***

## 六、建議補充的 Calculated Column（`dim_date`）

`dim_date` 需確保以下欄位存在，供 Slicer 與 Axis 使用：

```dax
-- 月份短名（Slicer / Axis）
month_name_short = FORMAT('dim_date'[date_key], "MMM")

-- 月份排序
month_sort = MONTH('dim_date'[date_key])

-- 年月組合（Trend Axis）
year_month = FORMAT('dim_date'[date_key], "YYYY-MM")

-- 季度
quarter = "Q" & QUARTER('dim_date'[date_key])
```

設定 `month_name_short` 的 "Sort by Column" = `month_sort`，否則月份軸會按字母排成 Apr / Aug / Dec...。

***

## 七、Measure Group 建議清單（`_Measures` 表）

在 Power BI 的 Model View 用 **Display Folder** 組織所有 Measures，方便日後維護：

| Display Folder | Measures |
|---|---|
| `01_Base` | Total Revenue, Total Sales Qty, Total COGS, Beginning/Ending Inventory Value, Current Stock Qty, Average Inventory Value |
| `02_Turnover` | Inventory Turnover, Days Sales of Inventory (DSI), Inventory Turnover YTD |
| `03_Stockout` | Out of Stock Records, Total Snapshot Records, Stockout Rate |
| `04_Dead Stock` | Sales Qty Last 90 Days, Dead Stock Value (90 Days), Dead Stock SKU Count, Dead Stock % of Total Inventory, Dead Stock Flag |
| `05_Reorder` | Avg Daily Sales Qty, Stddev Daily Sales Qty, Safety Stock Qty, Reorder Point Qty, Reorder Alert, Reorder SKU Count |
| `06_Unclassified` | Unclassified SKU Count, Unclassified Inventory Value, Unclassified Dead Stock Value |
| `07_Titles` | Title - Dead Stock by ABC, Title - Reorder Matrix |

***

## 八、作品集說明文字建議（README / 報表 Info Button）

為每個 KPI 加上「i」按鈕 Tooltip，說明計算邏輯與資料假設，能展示對方法論的掌握。建議文字如下：

**Inventory Turnover**：COGS / 平均庫存值（期初+期末除以2）。本資料集庫存快照僅有期初期末兩點，故此指標為年度估算。

**DSI**：365 / Inventory Turnover。同上，為年度層級最準確。

**Stockout Rate**：庫存量 ≤ 0 的快照記錄數 / 總快照記錄數。以快照比例定義，非以 SKU 天數。

**Dead Stock**：過去 90 天零銷售 且 期末庫存 > 0 的 SKU 庫存金額。90 天窗口自最後一筆銷售日動態往前推算。

**Reorder Point**：(日均銷售量 × 前置天數) + 安全庫存。安全庫存採 Z=1.65（95% 服務水準）× 銷售標準差 × √前置天數（7 天）。前置天數預設為 7 天，無原始資料依據，屬業界假設。

**ABC Classification**：依全年銷售額計算累積佔比，前 80% 為 A、80–95% 為 B、後 5% 為 C。在 PostgreSQL 預計算後寫入 `dim_product.abc_class`，避免 Power BI 對 1,200 萬筆 fact 表進行 Running Total 導致效能問題。



